In [1]:
# Installations

! pip install google-generativeai langchain --quiet
! pip install pypdf --quiet # required for LangChain pdf reader

### Agent related imports

In [2]:
from langchain.agents import load_tools
from langchain.agents import initialize_agent
from langchain.agents import AgentType

### LLM - Google PaLm2

In [3]:
from langchain.llms import GooglePalm

llm = GooglePalm(google_api_key='YOUR_PALM2_KEY',
                 temperature=0.0,
                top_k=1)

### Loading a tool

In [4]:
tools = load_tools(["llm-math"], llm=llm)

In [5]:
dir(AgentType)

['CHAT_CONVERSATIONAL_REACT_DESCRIPTION',
 'CHAT_ZERO_SHOT_REACT_DESCRIPTION',
 'CONVERSATIONAL_REACT_DESCRIPTION',
 'OPENAI_FUNCTIONS',
 'OPENAI_MULTI_FUNCTIONS',
 'REACT_DOCSTORE',
 'SELF_ASK_WITH_SEARCH',
 'STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION',
 'ZERO_SHOT_REACT_DESCRIPTION',
 '__class__',
 '__doc__',
 '__members__',
 '__module__']

### Agent Creation

In [6]:
agent = initialize_agent(tools, 
                         llm, 
                         agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, 
                         verbose=True)

### Interaction with the Agent

In [7]:
agent.run("What is 365 times 70?")



> Entering new AgentExecutor chain...
I need to multiply 365 by 70
Action: Calculator
Action Input: 365 * 70
Observation: Answer: 25550
Thought:I now know the final answer
Final Answer: 25550

> Finished chain.


'25550'

In [8]:
365*70

25550

### Adding multiple tools to the Agent - wikipedia + llm-math

In [9]:
! pip install wikipedia --q

In [10]:
tools = load_tools(['wikipedia', "llm-math"], llm=llm)

agent = initialize_agent(
    tools=tools, 
    llm=llm, 
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, 
    return_intermediate_steps=True,
    verbose=True,
)

In [11]:
from langchain.callbacks import StdOutCallbackHandler

prompt = """
How many runs Sachine Tendulkar scored? Also how many ODI hundreds he has scored? 
Return the runs multiplied by 1.5
"""

handler = StdOutCallbackHandler()
response = agent(
    {"input": prompt}, 
    callbacks=[handler]
)



> Entering new AgentExecutor chain...


> Entering new LLMChain chain...
Prompt after formatting:
Answer the following questions as best you can. You have access to the following tools:

Wikipedia: A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.
Calculator: Useful for when you need to answer questions about math.

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [Wikipedia, Calculator]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: 
How many runs Sachine Tendulkar scored? Also how many ODI hundreds he has scored? 


### Custom Tool Creation

In [12]:
from langchain.agents import tool

In [13]:
agent = initialize_agent(tools, 
                         llm, 
                         agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, 
                         verbose=True)

In [14]:
agent.run('Who is the best ODI batsman?')



> Entering new AgentExecutor chain...
I need to find out who the best ODI batsman is
Action: Wikipedia
Action Input: best ODI batsman
Observation: Page: Babar Azam
Summary: Mohammad Babar Azam (pronounced [mʊɦəmməd baːbəɾ aːzəm]; born 15 October 1994), is a Pakistani international cricketer and the former captain of the Pakistan national cricket team in all formats. He is widely regarded as one of the finest batters in contemporary world cricket. He is presently the only cricketer in the world to be in the top 5 rankings across all formats and is currently ranked as the number two batter in ODIs, fourth in T20I and Tests. A right-handed top-order batter, he captains Peshawar Zalmi in the PSL.With 42 wins, he is the joint most successful T20I captain of all time.He is Pakistan's most prolific T20I batter and his maiden T20I century of 122, which he scored against South Africa, is the highest current individual score by a Pakistani, in the format. He also remains the only captain to ta

'Virat Kohli'

### Creating custom tool with python function + doc string

In [15]:
@tool
def best_odi_batsman(text: str) -> str:
    '''Returns the name of Best ODI batsman of all time'''
    return "Sachine Tendulkar"

In [16]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from '/opt/conda/lib/python3.10/site-packages/wikipedia/__init__.py'>, top_k_results=3, lang='en', load_all_available_meta=False, doc_content_chars_max=4000)),
 Tool(name='Calculator', description='Useful for when you need to answer questions about math.', func=<bound method Chain.run of LLMMathChain(llm_chain=LLMChain(prompt=PromptTemplate(input_variables=['question'], template='Translate a math problem into a expression that can be executed using Python\'s numexpr library. Use the output of running this code to answer the question.\n\nQuestion: ${{Question with math problem.}}\n```text\n${{single line mathematical expression that solves the problem}}\n```\n...numexpr.evaluate(text)...\n```output\n${{Output of running the code}}\n```\nAnswer: ${{Answer}}\n\nBegin.\n\nQuestion: What is 37593 * 67?\n```text\n37593 * 67\n```\n...numexpr.evaluate("37593 * 67")...\n```output\n2518731\n```\nAnswer: 2518731\n\

In [17]:
tools = tools +[best_odi_batsman]
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from '/opt/conda/lib/python3.10/site-packages/wikipedia/__init__.py'>, top_k_results=3, lang='en', load_all_available_meta=False, doc_content_chars_max=4000)),
 Tool(name='Calculator', description='Useful for when you need to answer questions about math.', func=<bound method Chain.run of LLMMathChain(llm_chain=LLMChain(prompt=PromptTemplate(input_variables=['question'], template='Translate a math problem into a expression that can be executed using Python\'s numexpr library. Use the output of running this code to answer the question.\n\nQuestion: ${{Question with math problem.}}\n```text\n${{single line mathematical expression that solves the problem}}\n```\n...numexpr.evaluate(text)...\n```output\n${{Output of running the code}}\n```\nAnswer: ${{Answer}}\n\nBegin.\n\nQuestion: What is 37593 * 67?\n```text\n37593 * 67\n```\n...numexpr.evaluate("37593 * 67")...\n```output\n2518731\n```\nAnswer: 2518731\n\

In [18]:
agent = initialize_agent(tools, 
                         llm, 
                         agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, 
                         verbose=True)

In [19]:
agent.run('Who is the best ODI batsman?')



> Entering new AgentExecutor chain...
I need to find the best ODI batsman
Action: best_odi_batsman
Action Input: 
Observation: Sachine Tendulkar
Thought:I now know the final answer
Final Answer: Sachine Tendulkar

> Finished chain.


'Sachine Tendulkar'